In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import gc
import struct
import polars as pl
import numpy as np

In [2]:
def get_halfkp_indices(fen: str):
    # Fast FEN parsing to extract piece placement and king positions
    parts = fen.split(' ')
    placement = parts[0]
    
    rows = placement.split('/')
    pieces = []  # list of (sq, p_type, is_piece_white)
    white_king_sq = None
    black_king_sq = None
    
    sq = 56  # start at a8 (row 8, col 0)
    for row in rows:
        for char in row:
            if char.isdigit():
                sq += int(char)
            else:
                is_piece_white = char.isupper()
                char_lower = char.lower()
                
                # Determine piece type: P=1, N=2, B=3, R=4, Q=5, K=6
                if char_lower == 'p':
                    p_type = 1
                elif char_lower == 'n':
                    p_type = 2
                elif char_lower == 'b':
                    p_type = 3
                elif char_lower == 'r':
                    p_type = 4
                elif char_lower == 'q':
                    p_type = 5
                elif char_lower == 'k':
                    p_type = 6
                    if is_piece_white:
                        white_king_sq = sq
                    else:
                        black_king_sq = sq
                else:
                    sq += 1
                    continue
                
                if p_type != 6:
                    pieces.append((sq, p_type, is_piece_white))
                sq += 1
        sq -= 16

    # Calculate white perspective indices
    white_indices = []
    if white_king_sq is not None:
        for sq, p_type, is_piece_white in pieces:
            p_idx = p_type - 1 if is_piece_white else p_type + 4
            idx = sq + p_idx * 64 + white_king_sq * 640
            white_indices.append(idx)
        
    # Calculate black perspective indices
    black_indices = []
    if black_king_sq is not None:
        flipped_black_king_sq = black_king_sq ^ 56
        for sq, p_type, is_piece_white in pieces:
            flipped_sq = sq ^ 56
            p_idx = p_type - 1 if not is_piece_white else p_type + 4
            idx = flipped_sq + p_idx * 64 + flipped_black_king_sq * 640
            black_indices.append(idx)
        
    return white_indices, black_indices

In [3]:
class NnueDataset(Dataset):
    def __init__(self, pl_df):
        self.data = pl_df 
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        fen, score, result = self.data.row(idx)

        white_indices, black_indices = get_halfkp_indices(fen)

        # Pad indices to a fixed size of 32 to allow DataLoader collation
        padding_idx = 40960
        white_indices = (white_indices + [padding_idx] * 32)[:32]
        black_indices = (black_indices + [padding_idx] * 32)[:32]

        return {
            'white_indices': torch.tensor(white_indices, dtype=torch.long),
            'black_indices': torch.tensor(black_indices, dtype=torch.long),
            'eval_score': torch.tensor([score], dtype=torch.float32),
            'game_result': torch.tensor([result], dtype=torch.float32)
        }

In [4]:
class ClippedReLU(nn.Module):
    def __init__(self, max_val=1.0):
        super().__init__()
        self.max_val = max_val
    
    def forward(self, x):
        return torch.clamp(x, min=0, max=self.max_val)

In [5]:
class NNUE(nn.Module):
    def __init__(self):
        super().__init__()
        
        # input size is 40961, maximum idx is 40959, idx 40960 is padding idx
        self.embedding_layer = nn.EmbeddingBag(num_embeddings=40961, embedding_dim=256, mode='sum', padding_idx=40960)
        
        self.hidden_layer = nn.Linear(512, 64)
        self.crelu = ClippedReLU()
        self.output_layer = nn.Linear(64, 1)
        
    def forward(self, white_indices, black_indices):
        # Activation = ClippedReLU(sum of active Weights)
        white_emb = self.embedding_layer(white_indices)
        black_emb = self.embedding_layer(black_indices)
        
        w_acc = self.crelu(white_emb)
        b_acc = self.crelu(black_emb)
        
        # hidden layer
        x = torch.cat([w_acc, b_acc], dim=1)
        x = self.hidden_layer(x)
        x = self.crelu(x)
        
        # output layer
        x = self.output_layer(x)
        
        return x

In [6]:
def export_nnue_to_binary(model, filepath):
    # Unpack model if wrapped in DataParallel
    if isinstance(model, torch.nn.DataParallel):
        model = model.module
        
    with open(filepath, 'wb') as f:
        input_weights = model.embedding_layer.weight.detach().cpu().numpy().flatten()
        
        # Write to file using struct. '<f' means Little-Endian Float32
        for weight in input_weights:
            f.write(struct.pack('<f', weight))
            
        hidden_weights = model.hidden_layer.weight.detach().cpu().numpy().flatten()
        hidden_biases = model.hidden_layer.bias.detach().cpu().numpy().flatten()
        
        for weight in hidden_weights:
            f.write(struct.pack('<f', weight))
        for bias in hidden_biases:
            f.write(struct.pack('<f', bias))
            
        output_weights = model.output_layer.weight.detach().cpu().numpy().flatten()
        output_bias = model.output_layer.bias.detach().cpu().numpy().flatten()
        
        for weight in output_weights:
            f.write(struct.pack('<f', weight))
        for bias in output_bias:
            f.write(struct.pack('<f', bias))

    print("Export complete!")

In [7]:
file_path = "/kaggle/input/notebooks/jaswinreddym/nnue-data/nnue_dataset.csv"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [8]:
val_rows = 5e+6
batch_size = 65536
chunk_size = 5e+6
lr_rate = 1e-3
epochs = 10
scale_f = 400.0
lb = 0.8

In [9]:
model = NNUE().to(device)
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs for parallel training!")
    model = torch.nn.DataParallel(model)
model = model.to(device)
optimizer = optim.AdamW(model.parameters(), lr=lr_rate)

Using 2 GPUs for parallel training!


In [10]:
# Load validation set once before epoch loop
print("Loading validation data...", end=" ")
val_df = pl.read_csv(file_path, n_rows=int(val_rows))
val_dataset = NnueDataset(val_df)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4)
print(f"Validation set of size {len(val_df)} loaded.")

Loading validation data... Validation set of size 5000000 loaded.


In [11]:
best_val_loss = float('inf')

for epoch in range(epochs):
    print(f"Starting epoch {epoch+1}...")

    # Read csv in batches starting from the beginning
    reader = pl.read_csv_batched(file_path, batch_size=int(chunk_size))
    
    # Skip validation rows dynamically to avoid training on validation data
    skipped_rows = 0
    while skipped_rows < int(val_rows):
        batches = reader.next_batches(1)
        if not batches:
            break
        skipped_rows += len(batches[0])
    
    idx = 1
    # Reading the remaining chunks for training
    print("\tChunks Finished:", end=" ")
    while True:
        batches = reader.next_batches(1)
        if not batches:
            break

        chunk = batches[0]
        if idx % 100 == 0 or idx == 1:
            print(f"\n\tTraining on chunk {idx} of size {len(chunk)}...")
            print("\tChunks Finished:", end=" ")

        dataset = NnueDataset(chunk)
        dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=4)

        model.train()
        for batch in dataloader:
            white_indices = batch['white_indices'].to(device)
            black_indices = batch['black_indices'].to(device)
            eval_scores = batch['eval_score'].to(device)
            game_results = batch['game_result'].to(device)

            optimizer.zero_grad()
            outputs = model(white_indices, black_indices)

            eval_prob = torch.sigmoid(eval_scores / scale_f)
            target = lb * eval_prob + (1-lb) * game_results
            loss = F.mse_loss(outputs, target)

            loss.backward()
            optimizer.step()
            
        print(idx, end=" ")
            
        idx += 1
        del batches, chunk, dataset, dataloader
        gc.collect()
        torch.cuda.empty_cache()

    # Validation
    print("\tRunning validation...")
    model.eval()
    val_loss = 0.0
    val_batches = 0
    with torch.no_grad():
        for batch in val_dataloader:
            white_indices = batch['white_indices'].to(device)
            black_indices = batch['black_indices'].to(device)
            eval_scores = batch['eval_score'].to(device)
            game_results = batch['game_result'].to(device)

            outputs = model(white_indices, black_indices)

            eval_prob = torch.sigmoid(eval_scores / scale_f)
            target = lb * eval_prob + (1-lb) * game_results
            loss = F.mse_loss(outputs, target)

            val_loss += loss.item()
            val_batches += 1

    avg_val_loss = val_loss / val_batches if val_batches > 0 else 0.0
    print(f"\tEpoch {epoch+1} - Avg Validation Loss: {avg_val_loss:.6f}")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        export_nnue_to_binary(model, "neuralgambit.nnue")
        # Also saving standard PyTorch checkpoint just in case (unpacking DataParallel first)
        raw_model = model.module if isinstance(model, torch.nn.DataParallel) else model
        torch.save(raw_model.state_dict(), "best_model.pt")
        print(f"\tNew best validation loss: {best_val_loss:.6f}. Model weights saved to neuralgambit.nnue.")

Starting epoch 1...
	Chunks Finished: 
	Training on chunk 1 of size 267738...
	Chunks Finished: 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71 72 73 74 75 76 77 78 79 80 81 82 83 84 85 86 87 88 89 90 91 92 93 94 95 96 97 98 99 
	Training on chunk 100 of size 267887...
	Chunks Finished: 100 101 102 103 104 105 106 107 108 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125 126 127 128 129 130 131 132 133 134 135 136 137 138 139 140 141 142 143 144 145 146 147 148 149 150 151 152 153 154 155 156 157 158 159 160 161 162 163 164 165 166 167 168 169 170 171 172 173 174 175 176 177 178 179 180 181 182 183 184 185 186 187 188 189 190 191 192 193 194 195 196 197 198 199 
	Training on chunk 200 of size 267636...
	Chunks Finished: 200 201 202 203 204 205 206 207 208 209 210 211 212 213 214 215 216 217 218 219 220 221 222 223 